# Nexus — Colab GPU Model Lab

Run a stronger Qwen model in this notebook on a Google Colab GPU. Open the Runtime menu and choose **Change runtime type → GPU**, then run the cells from top to bottom. GPU types and access depend on current Colab availability; a GPU is not guaranteed, and hosted runtimes are temporary.

This gives Nexus a notebook chat backed by Ollama in the Colab virtual machine. It does **not** control your Zorin desktop or read your local files, Gmail, or Telegram. The notebook does not open a public tunnel, mount Google Drive, or ask for account credentials. Chat text is sent to the model running in this Colab session. Do not enter secrets or private data.

Default model: **Qwen 3.5 9B** (about 6.6–7.6 GB in Ollama). If the assigned GPU runs out of memory, change the next cell to `qwen3.5:4b` and rerun the model download. Ollama can fall back to CPU when it cannot use the GPU, so inspect the GPU status cell after the first answer.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Qwen 3.5 models in Ollama](https://ollama.com/library/qwen3.5), [Ollama GPU support](https://github.com/ollama/ollama/blob/main/docs/gpu.mdx).

In [ ]:
# Pick qwen3.5:4b if the GPU assigned to your session has limited VRAM.
MODEL = "qwen3.5:9b"
NUM_CTX = 8192
NUM_PREDICT = 512
print("Selected model:", MODEL)


In [ ]:
# Confirm Colab assigned an NVIDIA GPU. If not, change Runtime type to GPU and reconnect.
import subprocess

gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(gpu_check.stdout)
else:
    print("No NVIDIA GPU is visible. You can still continue on CPU, but replies may be much slower.")
    print(gpu_check.stderr[-1200:])


In [ ]:
# Install Ollama from its official installer inside this temporary Colab VM.
import shutil

if shutil.which("ollama") is None:
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )
else:
    print("Ollama is already installed.")


In [ ]:
# Start the model server on localhost only; the notebook does not expose a public port.
import time
import urllib.error
import urllib.request

OLLAMA_URL = "http://127.0.0.1:11434"
ollama_process = None

def ollama_ready():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if not ollama_ready():
    log_file = open("/tmp/nexus-ollama.log", "w")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(60):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            break
        time.sleep(1)

if not ollama_ready():
    try:
        print(open("/tmp/nexus-ollama.log").read()[-3000:])
    except OSError:
        pass
    raise RuntimeError("Ollama did not start. Check the runtime, then rerun this cell.")

with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=5) as response:
    print("Ollama ready:", response.read().decode())


In [ ]:
# Download the selected model into this temporary Colab VM. This may take several minutes.
pull = subprocess.Popen(
    ["ollama", "pull", MODEL], stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in pull.stdout:
    print(line, end="")
if pull.wait() != 0:
    raise RuntimeError("Model download failed. Check the runtime connection and rerun this cell.")
print("Model is ready:", MODEL)


In [ ]:
# A plain-text notebook chat. Conversation history is held in RAM for this session only.
import json
import time
import ipywidgets as widgets
from IPython.display import display

SYSTEM_PROMPT = (
    "You are Nexus, a helpful English-speaking assistant. Be clear, thoughtful, "
    "and concise. This notebook cannot access or control the user's computer."
)
chat_messages = [{"role": "system", "content": SYSTEM_PROMPT}]


def ask_nexus(question):
    question = question.strip()
    if not question:
        return "Type a question first.", 0.0, 0.0
    chat_messages.append({"role": "user", "content": question})
    payload = {
        "model": MODEL,
        "messages": chat_messages,
        "stream": False,
        "keep_alive": "5m",
        "think": False,
        "options": {
            "num_ctx": NUM_CTX,
            "num_predict": NUM_PREDICT,
            "temperature": 0.2,
        },
    }
    request = urllib.request.Request(
        OLLAMA_URL + "/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    started = time.monotonic()
    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.loads(response.read())
    elapsed = time.monotonic() - started
    answer = result.get("message", {}).get("content", "").strip()
    chat_messages.append({"role": "assistant", "content": answer})
    stats = result.get("eval_count", 0) / max(result.get("eval_duration", 0) / 1e9, 1e-9)
    return answer, elapsed, stats

question_box = widgets.Textarea(
    placeholder="Ask Nexus something…",
    description="You:",
    layout=widgets.Layout(width="100%", height="80px"),
)
ask_button = widgets.Button(description="Ask Nexus", button_style="primary")
clear_button = widgets.Button(description="New chat")
chat_output = widgets.Output()


def on_ask(_):
    question = question_box.value.strip()
    if not question:
        return
    ask_button.disabled = True
    with chat_output:
        print("You:", question)
        try:
            answer, elapsed, tokens_per_second = ask_nexus(question)
            print("Nexus:", answer)
            print(f"\n{elapsed:.1f}s · {tokens_per_second:.1f} generated tokens/s · {MODEL}\n")
        except Exception as error:
            print("Request failed:", str(error))
    question_box.value = ""
    ask_button.disabled = False


def on_clear(_):
    chat_messages[:] = [{"role": "system", "content": SYSTEM_PROMPT}]
    chat_output.clear_output()
    question_box.value = ""

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)
display(widgets.VBox([
    question_box,
    widgets.HBox([ask_button, clear_button]),
    chat_output,
]))


In [ ]:
# After asking a question, check whether Ollama placed the model on the GPU.
subprocess.run(["ollama", "ps"], check=False)
subprocess.run(["nvidia-smi"], check=False)
